In [277]:
import os
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import pandas as pd
import pyaudioconvert as pac

import glob
import random

# from tflite_model_maker import audio_classifier
from IPython.display import Audio
from scipy.io import wavfile

In [278]:
BASEDIR = '/home/bhkim/github/bruit-classification'
DATADIR = os.path.join(BASEDIR, 'data')
MODELDIR = os.path.join(BASEDIR, 'model')

MODEL_HANDLE = 'https://tfhub.dev/google/yamnet/1'
SAMPLE_RATE = 16000
BATCH_SIZE = 1

In [279]:
for f in glob.glob(os.path.join(DATADIR, '**', '**', '*.wav')):
    if f.endswith('_16b.wav'):
        print(os.system(f'soxi {f}'))
    else:
        print(f)
        os.system(f'sox {f} -b 16 -r {SAMPLE_RATE} {f.replace(".wav","_16b.wav")}')

/home/bhkim/github/bruit-classification/data/val/background/62.wav

Input File     : '/home/bhkim/github/bruit-classification/data/val/background/32_16b.wav'
Channels       : 1
Sample Rate    : 16000
Precision      : 16-bit
Duration       : 00:00:05.07 = 81040 samples ~ 379.875 CDDA sectors
File Size      : 162k
Bit Rate       : 256k
Sample Encoding: 16-bit Signed Integer PCM

0

Input File     : '/home/bhkim/github/bruit-classification/data/val/background/2_16b.wav'
Channels       : 1
Sample Rate    : 16000
Precision      : 16-bit
Duration       : 00:00:02.13 = 34148 samples ~ 160.069 CDDA sectors
File Size      : 68.3k
Bit Rate       : 256k
Sample Encoding: 16-bit Signed Integer PCM

0
/home/bhkim/github/bruit-classification/data/val/background/52.wav
/home/bhkim/github/bruit-classification/data/val/background/22.wav

Input File     : '/home/bhkim/github/bruit-classification/data/val/background/12_16b.wav'
Channels       : 1
Sample Rate    : 16000
Precision      : 16-bit
Duration    

In [280]:
yamnet_model = hub.load(MODEL_HANDLE)

In [281]:
@tf.function
def load_wav_and_label(audio_file, label):
    # audio_data, _ = librosa.load(audio_file, sr=SAMPLE_RATE)
    # audio_data = tf.convert_to_tensor(audio_data)
    audio = tf.io.read_file(audio_file)
    audio, sample_rate = tf.audio.decode_wav(audio)
    audio = tf.squeeze(audio, axis=-1)
    return audio, label
    

In [282]:
train_audio_file_list = pd.Series(glob.glob(os.path.join(DATADIR, 'train/**/*_16b.wav')), dtype='object')
train_audio_label_list = [f.split('/')[-2] for f in train_audio_file_list]

val_audio_file_list = pd.Series(glob.glob(os.path.join(DATADIR, 'val/**/*_16b.wav')), dtype='object')
val_audio_label_list = [f.split('/')[-2] for f in val_audio_file_list]

test_audio_file_list = pd.Series(glob.glob(os.path.join(DATADIR, 'test/**/*_16b.wav')), dtype='object')
test_audio_label_list = [f.split('/')[-2] for f in test_audio_file_list]

In [283]:
class_name_dict = {}
i = 0
for label in train_audio_label_list:
    if not label in list(class_name_dict.keys()):
        class_name_dict[label] = i
        i += 1

In [284]:
train_audio_label_list = pd.Series([class_name_dict[i] for i in train_audio_label_list], dtype='int64')
val_audio_label_list = pd.Series([class_name_dict[i] for i in val_audio_label_list], dtype='int64')
test_audio_label_list = pd.Series([class_name_dict[i] for i in test_audio_label_list], dtype='int64')

In [285]:
train_ds = tf.data.Dataset.from_tensor_slices((train_audio_file_list, train_audio_label_list))
train_ds = train_ds.map(load_wav_and_label)

val_ds = tf.data.Dataset.from_tensor_slices((val_audio_file_list, val_audio_label_list))
val_ds = val_ds.map(load_wav_and_label)

test_ds = tf.data.Dataset.from_tensor_slices((test_audio_file_list, test_audio_label_list))
test_ds = test_ds.map(load_wav_and_label)

In [286]:
@tf.function
def extract_embedding(wav_data, label):
  scores, embeddings, spectrogram = yamnet_model(wav_data)
  num_embeddings = tf.shape(embeddings)[0]
  return embeddings, tf.repeat(label, num_embeddings)

In [287]:
train_ds = train_ds.map(extract_embedding)
val_ds = val_ds.map(extract_embedding)
test_ds = test_ds.map(extract_embedding)

In [288]:
train_ds = train_ds.cache().shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [289]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024), dtype=tf.float32, name='yamnet_embedding'),
    # tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(len(list(class_name_dict.keys())))
], name='bruit_yamnet')

model.summary()

Model: "bruit_yamnet"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_15 (Dense)            (None, 5)                 5125      
                                                                 
Total params: 5,125
Trainable params: 5,125
Non-trainable params: 0
_________________________________________________________________


In [290]:
model.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              optimizer=tf.keras.optimizers.Adam(
                  learning_rate=1e-4
                  ),
              metrics=['accuracy'])

callback = tf.keras.callbacks.EarlyStopping(monitor='loss',
                                            patience=3,
                                            restore_best_weights=True)

In [291]:
history = model.fit(train_ds,
                    epochs=30,
                    validation_data=val_ds,
                    callbacks=callback)

Epoch 1/30


126/145 [=========================>....] - ETA: 0s - loss: 1.7518 - accuracy: 0.2109   WARNING:tensorflow:Model was constructed with shape (None, 1024) for input KerasTensor(type_spec=TensorSpec(shape=(None, 1024), dtype=tf.float32, name='yamnet_embedding'), name='yamnet_embedding', description="created by layer 'yamnet_embedding'"), but it was called on an input with incompatible shape (None, None, 1024).


145/145 [==============================] - 6s 7ms/step - loss: 1.7195 - accuracy: 0.2282 - val_loss: 1.7018 - val_accuracy: 0.2237
Epoch 2/30
145/145 [==============================] - 0s 873us/step - loss: 1.5961 - accuracy: 0.2654 - val_loss: 1.6096 - val_accuracy: 0.2694
Epoch 3/30
145/145 [==============================] - 0s 903us/step - loss: 1.5279 - accuracy: 0.3140 - val_loss: 1.5665 - val_accuracy: 0.3653
Epoch 4/30
145/145 [==============================] - 0s 906us/step - loss: 1.4877 - accuracy: 0.3774 - val_loss: 1.5037 - val_accuracy: 0.4612
Epoch 5/30
145/145 [==============================] - 0s 887us/step - loss: 1.4442 - accuracy: 0.4078 - val_loss: 1.4702 - val_accuracy: 0.4932
Epoch 6/30
145/145 [==============================] - 0s 866us/step - loss: 1.4060 - accuracy: 0.4409 - val_loss: 1.4424 - val_accuracy: 0.5114
Epoch 7/30
145/145 [==============================] - 0s 884us/step - loss: 1.3726 - accuracy: 0.4558 - val_loss: 1.3941 - val_accuracy: 0.5342
Epoch

In [292]:
loss, accuracy = model.evaluate(test_ds)
print(loss)
print(accuracy)

47/47 [==============================] - 2s 40ms/step - loss: 1.1991 - accuracy: 0.4066
1.1990915536880493
0.4066317677497864


In [293]:
class ReduceMeanLayer(tf.keras.layers.Layer):
  def __init__(self, axis=0, **kwargs):
    super(ReduceMeanLayer, self).__init__(**kwargs)
    self.axis = axis

  def call(self, input):
    return tf.math.reduce_mean(input, axis=self.axis)

In [294]:
saved_model_path = os.path.join(MODELDIR)

input_segment = tf.keras.layers.Input(shape=(), dtype=tf.float32, name='audio')
embedding_extraction_layer = hub.KerasLayer(MODEL_HANDLE,
                                            trainable=False, name='yamnet')
_, embeddings_output, _ = embedding_extraction_layer(input_segment)
serving_outputs = model(embeddings_output)
serving_outputs = ReduceMeanLayer(axis=0, name='classifier')(serving_outputs)
serving_model = tf.keras.Model(input_segment, serving_outputs)
serving_model.save(saved_model_path, include_optimizer=False)

INFO:tensorflow:Assets written to: /home/bhkim/github/bruit-classification/model/assets


INFO:tensorflow:Assets written to: /home/bhkim/github/bruit-classification/model/assets
